In [1]:
import os
import sys
from pathlib import Path

import pandas as pd

In [2]:
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))

if project_root not in sys.path:
    sys.path.append(project_root)

In [3]:
from mlp_model.activations.sigmoid import Sigmoid
from mlp_model.initialize_weights.xavier import XavierInitializer
from mlp_model.layers.hidden import Hidden
from mlp_model.layers.output_layer import OutputLayer
from mlp_model.losses.mse_loss import MSELoss
from mlp_model.mlp import MLP
from mlp_model.optimizers.sgd import SGDOptimizer
from trainer.trainer import MLPTrainer

In [4]:
data_kind = "activation"
data_count = 1000 # 100, 500, 1000, 10000

train_filepath = Path(project_root) / "data/regression" / f"data.{data_kind}.train.{data_count}.csv"

In [5]:
df = pd.read_csv(train_filepath)
df.head(5)

,x,y
0,3.180650,-11.975601
1,4.509115,-3.256370
2,2.541621,-21.890366
3,-1.657136,-251.980642
4,-0.983170,-218.305912


In [6]:
df.shape

(1000, 2)

In [7]:
df.describe()

,x,y
count,1000.000000,1000.000000
mean,1.434247,-93.366188
std,2.016732,89.207275
min,-1.992891,-264.053804
25%,-0.369432,-177.356035
50%,1.384894,-60.054740
75%,3.161770,-12.168893
max,4.984651,-2.043882


In [8]:
x = df["x"]
y = df["y"]

In [9]:
x.head(5)

0    3.180650
1    4.509115
2    2.541621
3   -1.657136
4   -0.983170
Name: x, dtype: float64

In [10]:
y.head(5)

0    -11.975601
1     -3.256370
2    -21.890366
3   -251.980642
4   -218.305912
Name: y, dtype: float64

In [11]:
mlp = MLP(
    [
        Hidden(1, 16, Sigmoid()),
        Hidden(16, 32, Sigmoid()),
        Hidden(32, 64, Sigmoid()),
        OutputLayer(64, 1)
    ]
)
mlp.initialize_weights(XavierInitializer())

sgd = SGDOptimizer(mlp, 0.01)

In [12]:
x = df["x"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
y = df["y"].to_numpy().reshape(-1, 1)  # shape: (1000, 1)
trainer = MLPTrainer(mlp, sgd, MSELoss(), 500, 32)
results = trainer.cross_validate(x, y, 5)

Fold 1: loss = 0.022377 | metrics {'r2': 0.9999970700251295, 'mae': 0.12497464657992678, 'mse': 0.022376600929673272}
Fold 2: loss = 0.059309 | metrics {'r2': 0.999992305145832, 'mae': 0.196643936029689, 'mse': 0.059308997578326375}
Fold 3: loss = 0.646998 | metrics {'r2': 0.9999132216692395, 'mae': 0.6378867691344701, 'mse': 0.6469979337529557}
Fold 4: loss = 0.039454 | metrics {'r2': 0.9999952788511125, 'mae': 0.1655035283457657, 'mse': 0.03945442606479741}
Fold 5: loss = 0.072454 | metrics {'r2': 0.999991335375096, 'mae': 0.2064950371337698, 'mse': 0.0724543992842517}
---------------------------------------------------------------
Cross validation with 5 folds:
Mean loss: 0.168118 (std: 0.240047)
---------------------------------------------------------------
